Gold Layer: The Gold layer is the final and most important layer in a modern data pipeline (Bronze -> Silver -> Gold). It is where raw data is transformed into business-ready datasets that are optimized for analytics, dashboards, and decision-making.

#### Loading data from Silver layer

In [0]:
orders_df = spark.table('workspace.silver.orders')
customer_df = spark.table('workspace.silver.customers')
order_items_df = spark.table('workspace.silver.order_items')
products_df = spark.table('workspace.silver.products')
sellers_df = spark.table('workspace.silver.sellers')
order_review_df = spark.table('workspace.silver.order_reviews')
payment_df = spark.table('workspace.silver.order_payments')

#### Fact Tables

In [0]:
from pyspark.sql.functions import *
fact_order_items = order_items_df.join(orders_df.select('order_id','order_purchase_timestamp'), on = 'order_id', how = 'left')

fact_order_items = fact_order_items.withColumn('purchase_date_key',date_format("order_purchase_timestamp",'yyyyMMdd').cast('int'))

#fact_order_items.display()

In [0]:
fact_payment = payment_df.select('order_id','payment_sequential','payment_type','payment_installments','payment_value')

#display(fact_payment)

In [0]:
fact_reviews = order_review_df.select('review_id','order_id','review_score','review_comment_title','review_comment_message',
                                      'review_creation_date','review_answer_timestamp')
fact_reviews = fact_reviews.withColumn('review_creation_date_key',date_format("review_creation_date",'yyyyMMdd').cast('int'))
#display(fact_reviews)

In [0]:
fact_orders = orders_df.select('order_id','customer_id','order_status','order_delivered_carrier_date','order_delivered_customer_date',
                               'order_estimated_delivery_date','delay_days','delivery_days')
                               
fact_orders = orders_df.withColumn('purchase_date_key',date_format("order_purchase_timestamp",'yyyyMMdd').cast('int'))\
                        .withColumn('delivery_date_key',date_format("order_delivered_customer_date",'yyyyMMdd').cast('int'))\
                        .withColumn('estimated_delivery_date_key',date_format("order_estimated_delivery_date",'yyyyMMdd').cast('int'))

#display(fact_orders)

#### Dimension tables

In [0]:
dim_customer = customer_df.select('customer_id','customer_unique_id',
                                  'customer_zipcode','customer_city','customer_state',
                                  col('latitude').alias('customer_latitude'),
                                  col('longitude').alias('customer_longitude'))
#display(dim_customer)


    
dim_seller = sellers_df.select('seller_id',
                                  'seller_zipcode','seller_city','seller_state',
                                  col('latitude').alias('seller_latitude'),
                                  col('longitude').alias('seller_longitude'))
#display(dim_seller)

dim_product = products_df.select('product_id','product_category_name','product_weight_gm',
                                 'product_length_cm','product_height_cm','product_width_cm','product_photos_qty')
#display(dim_product)


In [0]:
dim_date = spark.sql("select explode(sequence(to_date('2016-01-01'), to_date('2018-12-31'), interval 1 day)) as full_date").select(
    date_format("full_date", "yyyyMMdd").cast('int').alias("date_key"),
    col("full_date"),
    year("full_date").alias("year"),
    month("full_date").alias("month"),
    quarter("full_date").alias("quarter"),
    dayofmonth("full_date").alias("day_of_month"),
    dayofweek("full_date").alias("day_of_week"),     
    date_format("full_date", "EEEE").alias("day_name"),
    date_format("full_date", "MMMM").alias("month_name"),
    weekofyear("full_date").alias("week_of_year"),
    when(dayofweek("full_date").isin([1, 7]), 1)
     .otherwise(0).alias("is_weekend"),
)


In [0]:
fact_order_items.select([sum(when(col(c).isNull(),1).otherwise(0)).alias(c) for c in fact_order_items.columns]).show()

#Dropping 8 order_item rows that has no match in orders as no parent order can't link to any customer, no purchase date can't do time analysis and no order_status can't filter by delivery status
fact_order_items = fact_order_items.filter(col("order_purchase_timestamp").isNotNull())

+--------+-------------+----------+---------+-------------------+-----+-------------+-----------+------------------------+-----------------+
|order_id|order_item_id|product_id|seller_id|shipping_limit_date|price|freight_value|total_price|order_purchase_timestamp|purchase_date_key|
+--------+-------------+----------+---------+-------------------+-----+-------------+-----------+------------------------+-----------------+
|       0|            0|         0|        0|                  0|    0|            0|          0|                       8|                8|
+--------+-------------+----------+---------+-------------------+-----+-------------+-----------+------------------------+-----------------+

